In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import  ChatOllama
from typing import TypedDict, Literal
from pydantic import BaseModel,Field

In [2]:
model = ChatOllama(model="llama3.2")

In [3]:
class SentimentSchema(BaseModel): 
    sentiment: Literal['positive', 'negative'] = Field(description="sentiment of the review.")

In [6]:
structured_model = model.with_structured_output(SentimentSchema)

In [8]:
prompt = "What is the sentiment of the following review - The software is bad"
structured_model.invoke(prompt)

SentimentSchema(sentiment='negative')

In [9]:
class ReviewState(TypedDict): 
    review: str 
    sentiment: Literal['positive', 'negative']
    diagnosi: dict 
    response: str

In [12]:
def find_sentiment(state: ReviewState):
    prompt =f"For the following review find out the sentiment \n {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

In [15]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)


#edges 
graph.add_edge(START, "find_sentiment")
graph.add_edge("find_sentiment", END)

workflow = graph.compile()

In [ ]:
inital_state = {
    "review": "The produce was really bad"
}
workflow.invoke(inital_state)

{'review': 'The producr was really bad', 'sentiment': 'negative'}